In [ ]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
import anndata
from PIL import Image

import src.loki.utils
import src.loki.preprocess

In [ ]:
data_dir = './loki_model'
model_path = os.path.join(data_dir, 'checkpoint.pt')
device = 'cpu'

In [ ]:
model, preprocess, tokenizer = src.loki.utils.load_model(model_path, device)

In [ ]:
model.eval()

In [ ]:
import torch
from tqdm import tqdm 
import os
# --- 1. 定义文件路径 ---
gene_file_path = 'select_genes/STNet_3D.npy'
output_dir = 'select_genes/'
output_filename = 'STNet_3D_loki_text_encode.npy'
output_filepath = os.path.join(output_dir, output_filename)

# 确保输出目录存在
os.makedirs(output_dir, exist_ok=True)

# --- 2. 读取基因名称 ---
print(f"正在从 {gene_file_path} 读取基因名称...")
try:
    gene_names = np.load(gene_file_path, allow_pickle=True).tolist()
    print(f"成功读取 {len(gene_names)} 个基因名称。")
except FileNotFoundError:
    print(f"错误：找不到基因文件 {gene_file_path}。请检查路径是否正确。")
    exit()

# --- 3. 逐一编码每个基因 ---
print("\n开始逐一编码每个基因...")

# 创建一个空列表，用于存储每个基因编码后的向量
all_gene_embeddings = []

with torch.no_grad():
    # 使用 for 循环遍历每一个基因名称
    # tqdm(gene_names) 会为循环创建一个进度条
    for gene_name in tqdm(gene_names, desc="正在编码基因"):
        
        # 重要：将单个基因名放入一个列表中，因为 encode_texts 函数可能期望接收一个列表
        text_input = [gene_name]
        
        # 调用编码函数，此时它只处理一个基因
        text_embedding = src.loki.utils.encode_texts(model, tokenizer, text_input, device)
        
        # 将得到的 1x768 维度的 PyTorch 张量转换为 NumPy 数组，并添加到列表中
        all_gene_embeddings.append(text_embedding.cpu().numpy())

# --- 4. 整合所有向量 ---
# 使用 np.vstack 将列表中的所有单个向量 (每个都是 1x768) 堆叠成一个大的数组
final_embeddings_array = np.vstack(all_gene_embeddings)

print("\n所有基因已独立编码完成！")
print(f"最终的嵌入向量数组形状为: {final_embeddings_array.shape}")

# --- 5. 保存编码结果到文件 ---
print(f"正在将编码结果保存到: {output_filepath}")
np.save(output_filepath, final_embeddings_array)
print("文件保存成功！")

print("\n正在验证保存的文件...")
loaded_embeddings = np.load(output_filepath)
print(f"从文件中加载的数组形状为: {loaded_embeddings.shape}")
if np.array_equal(final_embeddings_array, loaded_embeddings):
    print("验证成功：保存和加载的数据完全一致。")
else:
    print("警告：验证失败，保存和加载的数据不一致。")